In [1]:

!pip install transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 matplotlib==3.10.0 numpy==2.0.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 43.1 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.17.0
    Uninstalling huggingface_hub-1.17.0:
      Successfully uninstalled huggingface_hub-

In [2]:
import os
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


In [13]:
from datasets import load_dataset

dataset = load_dataset(
    "databricks/databricks-dolly-15k"
)

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [16]:
print(dataset["train"][0])


{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [17]:
small_dataset = dataset["train"].select(range(500))

In [7]:
bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.float32
)

model = AutoModelForCausalLM.from_pretrained(
   "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [8]:
print(model.get_memory_footprint()/1e6)


4400.193664


In [9]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): 

In [10]:
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    # the rank of the adapter, the lower the fewer parameters you'll need to train
    r=8,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'],
)
model = get_peft_model(model, config)
model


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
              (k_proj): Linear(in_features=2048, out_features=256, bias=False)
              (v_proj): Linear(in_features=2048, out_features=256, bias=False)
              (o_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias

In [11]:
print(model.get_memory_footprint()/1e6)

4408.483968


In [12]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f'Trainable parameters:      {train_p/1e6:.2f}M')
print(f'Total parameters:          {tot_p/1e6:.2f}M')
print(f'% of trainable parameters: {100*train_p/tot_p:.2f}%')


Trainable parameters:      2.07M
Total parameters:          1102.12M
% of trainable parameters: 0.19%


In [28]:
# converting dataset into conversational dataset
def format_dataset(examples):
    if isinstance(examples["instruction"], list):
        output_texts = []

        for i in range(len(examples["instruction"])):

            user_content = examples["instruction"][i]

            if examples["context"][i]:
                user_content += f"\n\nContext:\n{examples['context'][i]}"

            converted_sample = [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": examples["response"][i]},
            ]

            output_texts.append(converted_sample)

        return {"messages": output_texts}

    else:

        user_content = examples["instruction"]

        if examples["context"]:
            user_content += f"\n\nContext:\n{examples['context']}"

        converted_sample = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": examples["response"]},
        ]

        return {"messages": converted_sample}

In [32]:
small_dataset = small_dataset.map(
    format_dataset,
    batched=True
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [33]:
print(small_dataset.column_names)

['instruction', 'context', 'response', 'category', 'messages']


In [34]:
print(small_dataset[0]["messages"])

[{'content': "When did Virgin Australia start operating?\n\nContext:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'role': 'user'}, {'content': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'role': 'assistant'}]


In [37]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id
tokenizer.chat_template

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

"{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"

In [41]:
messages = small_dataset[0]["messages"]

print(messages)

[{'content': "When did Virgin Australia start operating?\n\nContext:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'role': 'user'}, {'content': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'role': 'assistant'}]


In [42]:
print(
    tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
)

<|user|>
When did Virgin Australia start operating?

Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.</s>
<|assistant|>
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.</s>



In [45]:
sft_config = SFTConfig(

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    gradient_accumulation_steps=1,
    per_device_train_batch_size=16,

    auto_find_batch_size=True,


    max_length=64,
    packing=True,
    packing_strategy='wrapped',
    num_train_epochs=10,
    learning_rate=3e-4,

    optim='paged_adamw_8bit',

    logging_steps=10,
    logging_dir='./logs',
    output_dir='./phi3-mini-yoda-adapter',
    report_to='none',
    bf16=torch.cuda.is_bf16_supported(including_emulation=False)
)


In [47]:
trainer = SFTTrainer(
    model=model.base_model.model,
    peft_config=config,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=small_dataset,
)

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [48]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [49]:
batch['input_ids'][0], batch['labels'][0]

(tensor([  264,   358, 12242,   287,   363,   278, 24124,   310,  2769,   975,
         10847,   322, 17750, 29936,   297, 22661,   322,  7766,  1199, 29892,
           278, 24124,   310,   278,   289,   473,   479, 11505,   975, 22182,
          1793,   322, 28238,  1927, 29889,     2, 29871,    13,   529, 29989,
          1792, 29989, 29958,    13, 15156,  6455,  4586,   515,   278,  1426,
          2367,   592,   263, 15837,   310,   278,  1667,  6273,   297, 15381,
           310,  8370,  1201,   337], device='cuda:0'),
 tensor([  264,   358, 12242,   287,   363,   278, 24124,   310,  2769,   975,
         10847,   322, 17750, 29936,   297, 22661,   322,  7766,  1199, 29892,
           278, 24124,   310,   278,   289,   473,   479, 11505,   975, 22182,
          1793,   322, 28238,  1927, 29889,     2, 29871,    13,   529, 29989,
          1792, 29989, 29958,    13, 15156,  6455,  4586,   515,   278,  1426,
          2367,   592,   263, 15837,   310,   278,  1667,  6273,   297, 153

In [51]:
def gen_prompt(tokenizer, sentence):
    converted_sample = [{"role": "user", "content": sentence}]
    prompt = tokenizer.apply_chat_template(
        converted_sample, tokenize=False, add_generation_prompt=True
    )
    return prompt


In [52]:
sentence = 'When did Virgin Australia start operating?'
prompt = gen_prompt(tokenizer, sentence)
print(prompt)

<|user|>
When did Virgin Australia start operating?</s>
<|assistant|>



In [58]:
from contextlib import nullcontext
def generate(model, tokenizer, prompt, max_new_tokens=64, skip_special_tokens=False):
    tokenized_input = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(model.device)

    model.eval()
    ctx = torch.autocast(device_type=model.device.type, dtype=model.dtype) \
          if model.dtype in [torch.float16, torch.bfloat16] else nullcontext()
    with ctx:
        gen_output = model.generate(**tokenized_input,
                                    eos_token_id=tokenizer.eos_token_id,
                                    max_new_tokens=max_new_tokens)

    output = tokenizer.batch_decode(gen_output, skip_special_tokens=skip_special_tokens)
    return output[0]


In [59]:
print(generate(model, tokenizer, prompt))


<|user|>
When did Virgin Australia start operating?</s> 
<|assistant|>
Virgin Australia started operating on 15 March 2000.</s>


In [60]:
trainer.save_model('TinyLlama-fineturning')
